# Week 3: Cited Paper Ranking Dataset for Method Evaluation

Constructs a cited paper ranking dataset using `priority_followed.csv` to test how embedding methods prioritize true references over random distractors.

**Pipeline:**
1. **Step 1** — Select 100 main papers (P_i) with 6–14 references from `priority_followed.csv`
2. **Step 2** — Build per-P_i evaluation set: n positives (actual cited papers) + 2n negatives (random)
3. **Step 3** — Rank 3n candidates per method; compute hit rate in top-n
4. **Step 4** — Aggregate metrics: Hit Rate %, Precision@n, MRR, nDCG@n

Caches:
- Embeddings → `data/embeddings_cache/week3/`
- Results → `data/results_cache/week3/`
- Outputs → `eval_dataset/week3/`

In [1]:
import os
import json
import pickle
import hashlib
import time
import random
import ast
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
print('Imports OK')

Imports OK


In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
FOLLOWED_CSV   = 'data/cleaned/priority_followed.csv'
PARSED_CACHE   = 'data/parsed_cache/followed_parsed.pkl'
EMBED_CACHE    = 'data/embeddings_cache/week3'
RESULTS_CACHE  = 'data/results_cache/week3'
EVAL_DIR       = 'eval_dataset/week3'

for d in [EMBED_CACHE, RESULTS_CACHE, EVAL_DIR, 'data/parsed_cache']:
    os.makedirs(d, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Methods to evaluate (remove 'gemini' if API budget is limited)
METHODS = ['bm25', 'tfidf', 'minilm', 'qwen', 'gemini']

# ── Model config (matches similarity_evaluation.ipynb) ─────────────────────
MAX_TEXT_CHARS = {
    'all-MiniLM-L6-v2':        1024,
    'allenai/specter2_base':    2048,
    'Qwen/Qwen3-Embedding-0.6B': 3200,
}
BATCH_SIZES = {
    'all-MiniLM-L6-v2':        256,
    'allenai/specter2_base':    128,
    'Qwen/Qwen3-Embedding-0.6B': 64,
}
MODEL_MAP = {
    'minilm':   'all-MiniLM-L6-v2',
    'specter2': 'allenai/specter2_base',
    'qwen':     'Qwen/Qwen3-Embedding-0.6B',
}

import torch
DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: mps


In [3]:
# ── Utility: text building ─────────────────────────────────────────────────

def safe_parse(val):
    if isinstance(val, (list, dict)):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except Exception:
            return val
    return val


def extract_concept_names(val):
    parsed = safe_parse(val)
    if isinstance(parsed, list):
        return ' '.join(c.get('display_name', '') for c in parsed if isinstance(c, dict))
    return ''


def build_text(row, fields=('abstract', 'title', 'concepts')):
    parts = []
    if 'title' in fields and isinstance(row.get('title'), str) and row['title']:
        parts.append(row['title'])
    if 'abstract' in fields and isinstance(row.get('abstract'), str) and row['abstract']:
        parts.append(row['abstract'])
    if 'concepts' in fields:
        ct = extract_concept_names(row.get('concepts', ''))
        if ct.strip():
            parts.append(ct)
    return ' '.join(parts)


# ── Utility: embedding cache ───────────────────────────────────────────────

def _corpus_hash(texts):
    return hashlib.md5(''.join(texts[:5]).encode()).hexdigest()[:8]


def _cache_path(model_name, prefix, n_texts, corpus_hash):
    safe = model_name.replace('/', '_').replace('-', '_')
    return os.path.join(EMBED_CACHE, f'{safe}_{prefix}_{n_texts}_{corpus_hash}.npy')


_MODEL_CACHE = {}

def _get_model(model_name):
    if model_name not in _MODEL_CACHE:
        print(f'Loading model: {model_name}')
        _MODEL_CACHE[model_name] = SentenceTransformer(model_name, device=DEVICE)
    return _MODEL_CACHE[model_name]


def _truncate_texts(texts, model_name):
    limit = MAX_TEXT_CHARS.get(model_name, 2048)
    return [t[:limit] for t in texts]


print('Utility functions OK')

Utility functions OK


In [4]:
# ── Similarity functions ───────────────────────────────────────────────────

def bm25_similarity(query_texts, corpus_texts):
    tokenized_corpus = [doc.lower().split() for doc in corpus_texts]
    bm25 = BM25Okapi(tokenized_corpus)
    scores = np.array([bm25.get_scores(q.lower().split()) for q in query_texts])
    positive = scores[scores > 0]
    k = float(np.median(positive)) if len(positive) > 0 else 1.0
    scores = np.where(scores > 0, scores / (scores + k), 0.0)
    return scores


def tfidf_similarity(query_texts, corpus_texts):
    vectorizer = TfidfVectorizer(max_features=50000, stop_words='english')
    all_texts = list(corpus_texts) + list(query_texts)
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    corpus_vectors = tfidf_matrix[:len(corpus_texts)]
    query_vectors  = tfidf_matrix[len(corpus_texts):]
    return cosine_similarity(query_vectors, corpus_vectors)


def sentence_embedding_similarity(query_texts, corpus_texts, model_name='all-MiniLM-L6-v2'):
    model = _get_model(model_name)
    batch_size = BATCH_SIZES.get(model_name, 64)

    corpus_texts_enc = _truncate_texts(corpus_texts, model_name)
    c_hash = _corpus_hash(corpus_texts_enc)
    corpus_cache = _cache_path(model_name, 'corpus', len(corpus_texts_enc), c_hash)

    if os.path.exists(corpus_cache):
        corpus_emb = np.load(corpus_cache)
    else:
        corpus_emb = model.encode(corpus_texts_enc, batch_size=batch_size,
                                  show_progress_bar=False, convert_to_numpy=True)
        np.save(corpus_cache, corpus_emb)

    query_texts_enc = _truncate_texts(query_texts, model_name)
    q_hash = _corpus_hash(query_texts_enc)
    query_cache = _cache_path(model_name, 'query', len(query_texts_enc), q_hash)

    if os.path.exists(query_cache):
        query_emb = np.load(query_cache)
    else:
        query_emb = model.encode(query_texts_enc, batch_size=batch_size,
                                 show_progress_bar=False, convert_to_numpy=True)
        np.save(query_cache, query_emb)

    return cosine_similarity(query_emb, corpus_emb)


def gemini_score_pair(query_text, candidate_text,
                      model_name='gemini-2.0-flash-lite'):
    prompt = (
        'Rate the semantic similarity between these two academic paper descriptions '
        'on a scale of 0 to 100, where 0 is completely unrelated and 100 is identical.\n\n'
        f'Paper 1:\n{query_text[:800]}\n\nPaper 2:\n{candidate_text[:800]}\n\n'
        'Respond with only a single integer.'
    )
    try:
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt)
        score = int(''.join(filter(str.isdigit, response.text.strip()))[:3])
        return min(max(score, 0), 100) / 100.0
    except Exception as e:
        print(f'  Gemini error: {e}')
        return 0.0


print('Similarity functions OK')

Similarity functions OK


In [5]:
# ── Load priority_followed data ────────────────────────────────────────────

if os.path.exists(PARSED_CACHE):
    print('Loading from parsed cache...')
    with open(PARSED_CACHE, 'rb') as f:
        followed_df = pickle.load(f)
else:
    print('Parsing priority_followed.csv (first time, may be slow)...')
    followed_df = pd.read_csv(FOLLOWED_CSV, low_memory=False)
    list_cols = ['referenced_works', 'related_works', 'concepts', 'keywords',
                 'authorships', 'topics', 'institutions']
    for col in list_cols:
        if col in followed_df.columns:
            followed_df[col] = followed_df[col].apply(safe_parse)
    with open(PARSED_CACHE, 'wb') as f:
        pickle.dump(followed_df, f)
    print('Saved to parsed cache.')

followed_df['text'] = followed_df.apply(lambda r: build_text(r), axis=1)

# Build id → row index lookup for fast reference resolution
id_to_idx = {row['id']: i for i, row in followed_df.iterrows()}

print(f'Loaded {len(followed_df):,} papers')
print(f'Columns with ref counts: referenced_works_count sample:', 
      followed_df['referenced_works_count'].dropna().describe())

Loading from parsed cache...
Loaded 9,946 papers
Columns with ref counts: referenced_works_count sample: count    9946.000000
mean       80.145184
std       116.013381
min         0.000000
25%        39.000000
50%        59.000000
75%        87.000000
max      4428.000000
Name: referenced_works_count, dtype: float64


## Step 1: Select 100 Main Papers (P_i)

Filter papers where `referenced_works_count ∈ [6, 14]` and at least 6 of their cited works are resolvable within the followed corpus. Sample 100 at random.

In [6]:
MAIN_PAPERS_PATH = os.path.join(EVAL_DIR, 'main_papers.json')

if os.path.exists(MAIN_PAPERS_PATH):
    print('Loading cached main_papers.json...')
    with open(MAIN_PAPERS_PATH) as f:
        main_papers = json.load(f)
    print(f'Loaded {len(main_papers)} main papers.')
else:
    print('Selecting main papers...')

    # Filter by ref count range
    mask = (followed_df['referenced_works_count'] >= 6) & \
           (followed_df['referenced_works_count'] <= 14)
    candidates = followed_df[mask].copy()
    print(f'Papers with 6-14 refs: {len(candidates):,}')

    # Further filter: need abstract text and ref list present
    def has_valid_refs(refs):
        return isinstance(refs, list) and len(refs) >= 6

    candidates = candidates[
        candidates['referenced_works'].apply(has_valid_refs) &
        candidates['abstract'].notna() &
        (candidates['abstract'].str.len() > 50)
    ]
    print(f'After quality filters: {len(candidates):,}')

    # Keep only those with at least MIN_IN_CORPUS refs resolvable in the followed corpus
    MIN_IN_CORPUS = 2
    def count_resolvable(refs):
        if not isinstance(refs, list):
            return 0
        return sum(1 for r in refs if r in id_to_idx)

    candidates['n_resolvable'] = candidates['referenced_works'].apply(count_resolvable)
    candidates = candidates[candidates['n_resolvable'] >= MIN_IN_CORPUS]
    print(f'After resolvability filter (>={MIN_IN_CORPUS} refs in corpus): {len(candidates):,}')

    sampled = candidates.sample(n=min(100, len(candidates)), random_state=RANDOM_SEED)

    main_papers = []
    for _, row in sampled.iterrows():
        refs = row['referenced_works'] if isinstance(row['referenced_works'], list) else []
        main_papers.append({
            'id': row['id'],
            'title': row.get('title', ''),
            'abstract': row.get('abstract', ''),
            'referenced_works': refs,
            'referenced_works_count': int(row['referenced_works_count']),
            'text': row['text'],
        })

    with open(MAIN_PAPERS_PATH, 'w') as f:
        json.dump(main_papers, f, indent=2)
    print(f'Saved {len(main_papers)} main papers to {MAIN_PAPERS_PATH}')

Selecting main papers...
Papers with 6-14 refs: 256
After quality filters: 167
After resolvability filter (>=2 refs in corpus): 11
Saved 11 main papers to eval_dataset/week3/main_papers.json


## Step 2: Build Per-P_i Evaluation Sets

For each P_i with n references:
- **Positives (n):** actual cited papers — looked up in corpus, or fetched via OpenAlex API
- **Negatives (2n):** random papers from `priority_followed.csv`, excluding P_i's refs and same-author works

Result: 3n candidates per query.

In [7]:
# ── OpenAlex API fetch for references not in corpus ────────────────────────
import urllib.request

def fetch_openalex_works(openalex_ids, batch_size=50, sleep=0.3):
    """Fetch work metadata for a list of OpenAlex IDs via public API."""
    results = {}
    ids = list(openalex_ids)
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i + batch_size]
        # Strip URL prefix if present
        short_ids = [b.split('/')[-1] for b in batch]
        filter_str = '|'.join(short_ids)
        url = f'https://api.openalex.org/works?filter=openalex_id:{filter_str}&per_page={batch_size}&select=id,title,abstract_inverted_index,concepts'
        try:
            with urllib.request.urlopen(url, timeout=15) as resp:
                data = json.loads(resp.read())
            for work in data.get('results', []):
                wid = work.get('id', '')
                abstract = ''
                inv = work.get('abstract_inverted_index')
                if isinstance(inv, dict):
                    positions = {}
                    for word, pos_list in inv.items():
                        for p in pos_list:
                            positions[p] = word
                    abstract = ' '.join(positions[k] for k in sorted(positions))
                results[wid] = {
                    'id': wid,
                    'title': work.get('title', '') or '',
                    'abstract': abstract,
                    'concepts': work.get('concepts', []),
                }
        except Exception as e:
            print(f'  API error for batch {i//batch_size}: {e}')
        time.sleep(sleep)
    return results


print('OpenAlex fetch function ready')

OpenAlex fetch function ready


In [8]:
EVAL_SETS_PATH = os.path.join(RESULTS_CACHE, 'eval_sets.pkl')

if os.path.exists(EVAL_SETS_PATH):
    print('Loading cached eval sets...')
    with open(EVAL_SETS_PATH, 'rb') as f:
        eval_sets = pickle.load(f)
    print(f'Loaded {len(eval_sets)} eval sets.')
else:
    print('Building eval sets...')

    # Collect all ref IDs not in corpus so we can batch-fetch them
    all_missing_ids = set()
    for paper in main_papers:
        for ref_id in paper['referenced_works']:
            if ref_id not in id_to_idx:
                all_missing_ids.add(ref_id)

    print(f'Fetching {len(all_missing_ids)} reference papers not in corpus via OpenAlex...')
    fetched_works = fetch_openalex_works(all_missing_ids)
    print(f'Fetched {len(fetched_works)} works from OpenAlex API')

    # Build eval sets
    all_followed_ids = set(followed_df['id'].tolist())

    eval_sets = []
    for paper in tqdm(main_papers, desc='Building eval sets'):
        pi_id   = paper['id']
        pi_text = paper['text']
        ref_ids = paper['referenced_works']

        # Build positive candidates
        positives = []
        for ref_id in ref_ids:
            if ref_id in id_to_idx:
                row = followed_df.iloc[id_to_idx[ref_id]]
                positives.append({
                    'id': ref_id,
                    'title': row.get('title', '') or '',
                    'abstract': row.get('abstract', '') or '',
                    'text': row['text'],
                    'source': 'corpus',
                })
            elif ref_id in fetched_works:
                w = fetched_works[ref_id]
                row_dict = {'title': w['title'], 'abstract': w['abstract'],
                            'concepts': w['concepts']}
                positives.append({
                    'id': ref_id,
                    'title': w['title'],
                    'abstract': w['abstract'],
                    'text': build_text(row_dict),
                    'source': 'openalex_api',
                })

        # Skip if not enough positives with usable text
        positives = [p for p in positives if len(p['text']) > 20]
        n = len(positives)
        if n < 2:
            continue

        # Build negative candidates: 2n random, excluding refs and same-author works
        ref_id_set = set(ref_ids)

        # Get author IDs for P_i to exclude same-author works
        pi_row = followed_df[followed_df['id'] == pi_id]
        pi_author_ids = set()
        if not pi_row.empty:
            auths = pi_row.iloc[0].get('authorships', [])
            if isinstance(auths, list):
                for a in auths:
                    if isinstance(a, dict):
                        aid = a.get('author', {}).get('id', '') if isinstance(a.get('author'), dict) else ''
                        if aid:
                            pi_author_ids.add(aid)

        exclude_ids = ref_id_set | {pi_id}
        neg_pool = followed_df[
            ~followed_df['id'].isin(exclude_ids) &
            (followed_df['text'].str.len() > 20)
        ]
        neg_sample = neg_pool.sample(n=min(2 * n, len(neg_pool)),
                                     random_state=hash(pi_id) % (2**31))
        negatives = [{
            'id': row['id'],
            'title': row.get('title', '') or '',
            'abstract': row.get('abstract', '') or '',
            'text': row['text'],
            'source': 'negative',
        } for _, row in neg_sample.iterrows()]

        eval_sets.append({
            'pi_id':       pi_id,
            'pi_title':    paper['title'],
            'pi_text':     pi_text,
            'n_positives': n,
            'positives':   positives,
            'negatives':   negatives,
        })

    with open(EVAL_SETS_PATH, 'wb') as f:
        pickle.dump(eval_sets, f)
    print(f'Built and cached {len(eval_sets)} eval sets → {EVAL_SETS_PATH}')

# Summary
n_pos_list = [e['n_positives'] for e in eval_sets]
n_neg_list = [len(e['negatives']) for e in eval_sets]
print(f'\nEval sets: {len(eval_sets)}')
print(f'Positives per query — mean: {np.mean(n_pos_list):.1f}, range: [{min(n_pos_list)}, {max(n_pos_list)}]')
print(f'Negatives per query — mean: {np.mean(n_neg_list):.1f}')

Building eval sets...
Fetching 85 reference papers not in corpus via OpenAlex...
Fetched 79 works from OpenAlex API


Building eval sets: 100%|██████████| 11/11 [00:00<00:00, 36.97it/s]

Built and cached 11 eval sets → data/results_cache/week3/eval_sets.pkl

Eval sets: 11
Positives per query — mean: 9.4, range: [6, 14]
Negatives per query — mean: 18.7


## Step 3: Compute Embeddings and Rankings

For each method, rank all 3n candidates per query and compute per-query hit rate:
```
hit_rate = (# true references in top-n) / n × 100%
```

In [9]:
# ── Ranking function ───────────────────────────────────────────────────────

def rank_candidates(query_text, candidates, method):
    """
    Rank `candidates` (list of dicts with 'text' key) against `query_text`.
    Returns list of (candidate_idx, score) sorted by score descending.
    """
    texts = [c['text'] for c in candidates]

    if method == 'bm25':
        scores = bm25_similarity([query_text], texts)[0]

    elif method == 'tfidf':
        scores = tfidf_similarity([query_text], texts)[0]

    elif method in MODEL_MAP:
        model_name = MODEL_MAP[method]
        sim_matrix = sentence_embedding_similarity([query_text], texts, model_name=model_name)
        scores = sim_matrix[0]

    elif method == 'gemini':
        scores = []
        for text in texts:
            score = gemini_score_pair(query_text, text)
            scores.append(score)
            time.sleep(0.5)  # rate limit
        scores = np.array(scores)

    else:
        raise ValueError(f'Unknown method: {method}')

    ranked = sorted(enumerate(scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked


print('Ranking function ready')

Ranking function ready


In [10]:
# ── Run rankings for all methods ───────────────────────────────────────────
# Results format: {method: [{pi_id, pi_title, n_positives, ranked, positive_indices}, ...]}

RANKINGS_CACHE = os.path.join(RESULTS_CACHE, 'rankings.pkl')

if os.path.exists(RANKINGS_CACHE):
    with open(RANKINGS_CACHE, 'rb') as f:
        all_rankings = pickle.load(f)
    print(f'Loaded rankings cache. Methods present: {list(all_rankings.keys())}')
else:
    all_rankings = {}

methods_to_run = [m for m in METHODS if m not in all_rankings]
print(f'Methods to compute: {methods_to_run}')

for method in methods_to_run:
    print(f'\n── Method: {method} ──')
    method_results = []

    for ev in tqdm(eval_sets, desc=method):
        candidates = ev['positives'] + ev['negatives']
        positive_ids = {p['id'] for p in ev['positives']}
        positive_indices = set(range(len(ev['positives'])))  # positives are first

        ranked = rank_candidates(ev['pi_text'], candidates, method)

        method_results.append({
            'pi_id':            ev['pi_id'],
            'pi_title':         ev['pi_title'],
            'n_positives':      ev['n_positives'],
            'n_candidates':     len(candidates),
            'ranked':           ranked,          # [(candidate_idx, score), ...]
            'positive_indices': positive_indices, # set of indices that are positives
        })

    all_rankings[method] = method_results

    # Save incrementally
    with open(RANKINGS_CACHE, 'wb') as f:
        pickle.dump(all_rankings, f)
    print(f'  Saved {len(method_results)} results for {method}')

print('\nAll rankings computed.')

Methods to compute: ['bm25', 'tfidf', 'minilm', 'qwen', 'gemini']

── Method: bm25 ──


bm25: 100%|██████████| 11/11 [00:00<00:00, 281.50it/s]


  Saved 11 results for bm25

── Method: tfidf ──


tfidf: 100%|██████████| 11/11 [00:00<00:00, 240.03it/s]


  Saved 11 results for tfidf

── Method: minilm ──


...

  Saved 11 results for minilm

── Method: qwen ──


qwen:   0%|          | 0/11 [00:00<?, ?it/s]

Loading model: Qwen/Qwen3-Embedding-0.6B


qwen:  27%|██▋       | 3/11 [01:11<03:10, 23.86s/it]


KeyboardInterrupt: 

## Step 4: Evaluation Metrics

Aggregate across all queries and report per method:

| Metric | Description |
|---|---|
| **Hit Rate %** | % of top-n slots filled by true references |
| **Precision@n** | Equivalent to mean hit rate |
| **MRR** | Mean reciprocal rank of first positive |
| **nDCG@n** | Ranking quality accounting for all positives' positions |

In [ ]:
# ── Metric functions ───────────────────────────────────────────────────────

def hit_rate_at_n(ranked, positive_indices, n):
    """Fraction of top-n retrieved that are true positives."""
    top_n = [idx for idx, _ in ranked[:n]]
    hits = sum(1 for idx in top_n if idx in positive_indices)
    return hits / n


def reciprocal_rank(ranked, positive_indices):
    """1/rank of first positive in ranked list."""
    for rank, (idx, _) in enumerate(ranked, start=1):
        if idx in positive_indices:
            return 1.0 / rank
    return 0.0


def ndcg_at_n(ranked, positive_indices, n):
    """nDCG@n with binary relevance."""
    def dcg(hits):
        return sum(h / math.log2(i + 2) for i, h in enumerate(hits))

    top_n_hits = [1 if idx in positive_indices else 0
                  for idx, _ in ranked[:n]]
    ideal_hits = sorted(top_n_hits, reverse=True)
    ideal_dcg = dcg(ideal_hits)
    if ideal_dcg == 0:
        return 0.0
    return dcg(top_n_hits) / ideal_dcg


def compute_metrics(method_results):
    """Return dict of metric_name → list of per-query values."""
    hit_rates, mrrs, ndcgs = [], [], []

    for res in method_results:
        n   = res['n_positives']
        pos = res['positive_indices']
        r   = res['ranked']

        hit_rates.append(hit_rate_at_n(r, pos, n) * 100)
        mrrs.append(reciprocal_rank(r, pos))
        ndcgs.append(ndcg_at_n(r, pos, n))

    return {
        'hit_rate': hit_rates,
        'mrr':      mrrs,
        'ndcg':     ndcgs,
    }


print('Metric functions ready')

In [ ]:
# ── Aggregate and display results ──────────────────────────────────────────

rows = []
metrics_by_method = {}

for method, results in all_rankings.items():
    m = compute_metrics(results)
    metrics_by_method[method] = m
    rows.append({
        'Method':              method,
        'Mean Hit Rate %':     round(np.mean(m['hit_rate']), 2),
        'Std Hit Rate %':      round(np.std(m['hit_rate']),  2),
        'Median Hit Rate %':   round(np.median(m['hit_rate']), 2),
        'Mean MRR':            round(np.mean(m['mrr']),  4),
        'Std MRR':             round(np.std(m['mrr']),   4),
        'Mean nDCG@n':         round(np.mean(m['ndcg']), 4),
        'Std nDCG@n':          round(np.std(m['ndcg']),  4),
    })

results_df = pd.DataFrame(rows).sort_values('Mean Hit Rate %', ascending=False)
print('\n=== Week 3 Results ===')
print(results_df.to_string(index=False))

# Save results table
results_df.to_csv(os.path.join(EVAL_DIR, 'metrics_summary.csv'), index=False)

# Save per-query metrics
with open(os.path.join(RESULTS_CACHE, 'metrics_by_method.pkl'), 'wb') as f:
    pickle.dump(metrics_by_method, f)

print(f'\nSaved metrics_summary.csv → {EVAL_DIR}')

In [ ]:
# ── Save per-method ranking files ──────────────────────────────────────────
# eval_dataset/week3/rankings_{method}.json

for method, results in all_rankings.items():
    m = metrics_by_method[method]
    output = []
    for i, res in enumerate(results):
        output.append({
            'pi_id':       res['pi_id'],
            'pi_title':    res['pi_title'],
            'n_positives': res['n_positives'],
            'hit_rate':    round(m['hit_rate'][i], 4),
            'mrr':         round(m['mrr'][i], 4),
            'ndcg':        round(m['ndcg'][i], 4),
            'ranked_ids':  [res['ranked'][j][0] for j in range(min(res['n_positives'], len(res['ranked'])))],
            'ranked_scores': [round(res['ranked'][j][1], 4) for j in range(min(res['n_positives'], len(res['ranked'])))],
        })
    path = os.path.join(EVAL_DIR, f'rankings_{method}.json')
    with open(path, 'w') as f:
        json.dump(output, f, indent=2)
    print(f'Saved {path}')

In [ ]:
# ── Visualisation ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

methods_sorted = results_df['Method'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Week 3: Cited Paper Ranking — Method Comparison', fontsize=14)

colors = plt.cm.tab10(np.linspace(0, 0.6, len(methods_sorted)))

# Hit Rate distribution
ax = axes[0]
data = [metrics_by_method[m]['hit_rate'] for m in methods_sorted]
ax.boxplot(data, labels=methods_sorted, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
ax.set_title('Hit Rate @ n (%)')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=30)

# MRR distribution
ax = axes[1]
data = [metrics_by_method[m]['mrr'] for m in methods_sorted]
ax.boxplot(data, labels=methods_sorted, patch_artist=True,
           boxprops=dict(facecolor='lightgreen'))
ax.set_title('MRR (Mean Reciprocal Rank)')
ax.tick_params(axis='x', rotation=30)

# nDCG@n distribution
ax = axes[2]
data = [metrics_by_method[m]['ndcg'] for m in methods_sorted]
ax.boxplot(data, labels=methods_sorted, patch_artist=True,
           boxprops=dict(facecolor='lightsalmon'))
ax.set_title('nDCG @ n')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
out_path = os.path.join(EVAL_DIR, 'method_comparison.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

In [ ]:
# ── Hit rate per query (sorted) ────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 5))

for method in methods_sorted:
    hr = sorted(metrics_by_method[method]['hit_rate'], reverse=True)
    ax.plot(hr, label=method, linewidth=1.5)

ax.set_title('Per-Query Hit Rate @ n (sorted descending)')
ax.set_xlabel('Query rank')
ax.set_ylabel('Hit Rate %')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(EVAL_DIR, 'per_query_hit_rate.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')